In [16]:
import numpy as np 
import pandas as pd 


import os
for dirname, _, filenames in os.walk('/kaggle/input/polarization'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/polarization/subtask2/train/tel.csv
/kaggle/input/polarization/subtask2/train/mya.csv
/kaggle/input/polarization/subtask2/train/amh.csv
/kaggle/input/polarization/subtask2/train/eng.csv
/kaggle/input/polarization/subtask2/train/ben.csv
/kaggle/input/polarization/subtask2/train/hau.csv
/kaggle/input/polarization/subtask2/train/khm.csv
/kaggle/input/polarization/subtask2/train/pan.csv
/kaggle/input/polarization/subtask2/train/pol.csv
/kaggle/input/polarization/subtask2/train/arb.csv
/kaggle/input/polarization/subtask2/train/hin.csv
/kaggle/input/polarization/subtask2/train/nep.csv
/kaggle/input/polarization/subtask2/train/ita.csv
/kaggle/input/polarization/subtask2/train/zho.csv
/kaggle/input/polarization/subtask2/train/urd.csv
/kaggle/input/polarization/subtask2/train/rus.csv
/kaggle/input/polarization/subtask2/train/swa.csv
/kaggle/input/polarization/subtask2/train/ori.csv
/kaggle/input/polarization/subtask2/train/deu.csv
/kaggle/input/polarization/subtask2/train/spa.csv


In [17]:
!pip install transformers pandas torch scikit-learn tqdm
!pip install sentencepiece

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
base_path = '/kaggle/input/polarization'
df_en = pd.read_csv(f'{base_path}/subtask1/train/eng.csv')
df_sw = pd.read_csv(f'{base_path}/subtask1/train/swa.csv')

In [19]:
df_en.head()

,id,text,polarization
0,eng_973938b90b0ff5d87d35a582f83f5c89,is defending imperialism in the dnd chat,0
1,eng_07dfd4600426caca6e2c5883fcbea9ea,Still playing with this. I am now following Ra...,0
2,eng_f14519ff2302b6cd47712073f13bc461,.senate.gov Theres 3 groups out there Republic...,0
3,eng_e48b7e7542faafa544ac57b64bc80daf,"""ABC MD, David Anderson, said the additional f...",0
4,eng_7c581fb77bce8033aeba3d6dbd6273eb,"""bad people"" I have some conservative values s...",0


In [20]:
print(df_en['polarization'].value_counts(normalize=True))
print(df_sw['polarization'].value_counts(normalize=True))

polarization
0    0.63532
1    0.36468
Name: proportion, dtype: float64
polarization
1    0.501216
0    0.498784
Name: proportion, dtype: float64


In [21]:
df_en = pd.read_csv(f'{base_path}/subtask2/train/eng.csv')
df_sw = pd.read_csv(f'{base_path}/subtask2/train/swa.csv')
df_sw.head()


,id,text,political,racial/ethnic,religious,gender/sexual,other
0,swa_53de6a7a4d0123b5755da79d8d97a82f,uwizi rt kenyan rao akishinda nitachinja kuku ...,0,1,0,0,0
1,swa_ee2533cb334df97236ea2bcfda0d6823,wakikuyu ndio wako na manyumba za kukodeshwa t...,0,1,0,0,0
2,swa_1dd81b5985840a55b1ab292aa65d11a8,wakikuyu ni wezi power hungry and this time we...,0,1,0,0,0
3,swa_18589adc3945e20c5e5c61e10245fad1,wakikuyu sijui shida yenu ni nini kuogopa rail...,0,1,0,0,0
4,swa_aee76fc4cd1c6c6c09e19ba5ddd3901a,wakikuyu walisogwa hwakuumbwa,0,1,0,0,0


In [22]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaTokenizer, XLMRobertaModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import gc
import torch.nn.functional as F

In [23]:
class Config:
    MAX_LEN = 128
    BATCH_SIZE = 8          
    EPOCHS = 4              
    LEARNING_RATE = 2e-5    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    LABELS_T1 = ['polarization']
    LABELS_T2 = ['political', 'racial/ethnic', 'religious', 'gender/sexual', 'other']
    LABELS_T3 = ['stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation']

In [24]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)  # prevents nans when probability 0
        focal_loss = self.alpha * (1-pt)**self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [25]:
def get_best_threshold(y_true, y_pred_probs):
    """
    Finds the best threshold for binary classification to maximize F1.
    Useful for the paper's 'Ablation Study' section.
    """
    best_f1 = 0
    best_thresh = 0.5
    
    # Search range from 0.1 to 0.9
    for thresh in np.arange(0.1, 0.91, 0.01):
        y_pred = (y_pred_probs >= thresh).astype(int)
        score = f1_score(y_true, y_pred, average='binary')
        if score > best_f1:
            best_f1 = score
            best_thresh = thresh
            
    return best_thresh, best_f1

In [26]:
def get_merged_train_data(lang):
    
    base_path = '/kaggle/input/polarization'
    
    
    df1 = pd.read_csv(f'{base_path}/subtask1/train/{lang}.csv')
    df2 = pd.read_csv(f'{base_path}/subtask2/train/{lang}.csv')
    df3 = pd.read_csv(f'{base_path}/subtask3/train/{lang}.csv')
    
    
    if 'text' in df2.columns: df2 = df2.drop(columns=['text'])
    if 'text' in df3.columns: df3 = df3.drop(columns=['text'])
    
    
    merged = df1.merge(df2, on='id', how='left').merge(df3, on='id', how='left')
    merged = merged.fillna(0) 
    return merged

def get_pos_weights(df, labels):
    
    weights = []
    for col in labels:
        pos = df[col].sum()
        neg = len(df) - pos
        weight = neg / (pos + 1e-5)
        weights.append(weight)
    return torch.tensor(weights, dtype=torch.float).to(Config.DEVICE)


In [27]:
# 3. DATASET & TOKENIZER
# ====================================================
class PolarizationDataset(Dataset):
    def __init__(self, df, tokenizer, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.is_test = is_test
        self.text = df['text'].astype(str).values
        self.ids = df['id'].values
        
        if not is_test:
            self.t1 = df[Config.LABELS_T1].values
            self.t2 = df[Config.LABELS_T2].values
            self.t3 = df[Config.LABELS_T3].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        text = " ".join(self.text[index].split()) 
        
        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=Config.MAX_LEN,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False
        )
        
        item = {
            'ids': torch.tensor(inputs['input_ids'], dtype=torch.long),
            'mask': torch.tensor(inputs['attention_mask'], dtype=torch.long),
            'id_val': self.ids[index]
        }
        
        if not self.is_test:
            item['t1_targets'] = torch.tensor(self.t1[index], dtype=torch.float)
            item['t2_targets'] = torch.tensor(self.t2[index], dtype=torch.float)
            item['t3_targets'] = torch.tensor(self.t3[index], dtype=torch.float)
            
        return item


In [28]:
# MEAN POOLING
# ====================================================
class AdvancedMultiTaskModel(nn.Module):
    def __init__(self, model_name):
        super(AdvancedMultiTaskModel, self).__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size
        self.drop = nn.Dropout(0.3)
        
        self.head1 = nn.Linear(self.hidden_size, 1) 
        self.head2 = nn.Linear(self.hidden_size, len(Config.LABELS_T2))
        self.head3 = nn.Linear(self.hidden_size, len(Config.LABELS_T3))

    def mean_pooling(self, model_output, attention_mask):
        
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, ids, mask):
        output = self.encoder(ids, attention_mask=mask)
        feature_vector = self.mean_pooling(output, mask)
        x = self.drop(feature_vector)
        
        return self.head1(x), self.head2(x), self.head3(x)


In [29]:
def run_kfold_pipeline(lang, model_name, n_folds=5):
    print(f"\n{'='*50}")
    print(f"START IMPROVED PIPELINE FOR: {lang.upper()}")
    print(f"MODEL: {model_name}")
    print(f"{'='*50}")
    
    # 1. Data Loading
    full_train_df = get_merged_train_data(lang)
    X = full_train_df['id'].values
    y = full_train_df['polarization'].values
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    # Test Data
    dev_path = f'/kaggle/input/polarization/subtask1/dev/{lang}.csv'
    df_dev = pd.read_csv(dev_path)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds_test = PolarizationDataset(df_dev, tokenizer, is_test=True)
    dl_test = DataLoader(ds_test, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    # Placeholders for Test Predictions
    test_len = len(df_dev)
    final_probs_t1 = np.zeros((test_len, 1))
    final_probs_t2 = np.zeros((test_len, len(Config.LABELS_T2)))
    final_probs_t3 = np.zeros((test_len, len(Config.LABELS_T3)))
    
    # Store Validation Results to calculate Best Thresholds later
    oof_preds_t1 = []
    oof_targets_t1 = []
    
    oof_preds_t2 = []
    oof_targets_t2 = []
    
    oof_preds_t3 = []
    oof_targets_t3 = []
    
    # --- FOLD LOOP ---
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n>>> Fold {fold + 1}/{n_folds}")
        
        train_df = full_train_df.iloc[train_idx].reset_index(drop=True)
        val_df = full_train_df.iloc[val_idx].reset_index(drop=True)
        
        train_ds = PolarizationDataset(train_df, tokenizer)
        val_ds = PolarizationDataset(val_df, tokenizer)
        
        train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False)
        
        model = AdvancedMultiTaskModel(model_name)
        model.to(Config.DEVICE)
        optimizer = AdamW(model.parameters(), lr=Config.LEARNING_RATE)
        
        # --- NEW LOSS: FOCAL LOSS ---
        # Gamma=2 focuses on hard examples
        crit = FocalLoss(alpha=1, gamma=2) 
        
        # Train
        model.train()
        for epoch in range(Config.EPOCHS):
            train_loss = 0
            for data in train_loader:
                ids = data['ids'].to(Config.DEVICE)
                mask = data['mask'].to(Config.DEVICE)
                
                optimizer.zero_grad()
                o1, o2, o3 = model(ids, mask)
                
                # Combine losses
                loss = crit(o1, data['t1_targets'].to(Config.DEVICE)) + \
                       crit(o2, data['t2_targets'].to(Config.DEVICE)) + \
                       crit(o3, data['t3_targets'].to(Config.DEVICE))
                
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            print(f"   Epoch {epoch+1} Loss: {train_loss/len(train_loader):.4f}")
            
        # --- VALIDATION (To find thresholds) ---
        model.eval()
        val_p1, val_p2, val_p3 = [], [], []
        val_t1, val_t2, val_t3 = [], [], []
        
        with torch.no_grad():
            for data in val_loader:
                ids = data['ids'].to(Config.DEVICE)
                mask = data['mask'].to(Config.DEVICE)
                
                o1, o2, o3 = model(ids, mask)
                
                val_p1.extend(torch.sigmoid(o1).cpu().numpy())
                val_p2.extend(torch.sigmoid(o2).cpu().numpy())
                val_p3.extend(torch.sigmoid(o3).cpu().numpy())
                
                val_t1.extend(data['t1_targets'].numpy())
                val_t2.extend(data['t2_targets'].numpy())
                val_t3.extend(data['t3_targets'].numpy())

        # Store OOF Data
        oof_preds_t1.append(val_p1)
        oof_targets_t1.append(val_t1)
        oof_preds_t2.append(val_p2)
        oof_targets_t2.append(val_t2)
        oof_preds_t3.append(val_p3)
        oof_targets_t3.append(val_t3)

        # --- PREDICT TEST ---
        fold_p1, fold_p2, fold_p3 = [], [], []
        with torch.no_grad():
            for data in dl_test:
                ids = data['ids'].to(Config.DEVICE)
                mask = data['mask'].to(Config.DEVICE)
                o1, o2, o3 = model(ids, mask)
                fold_p1.extend(torch.sigmoid(o1).cpu().numpy())
                fold_p2.extend(torch.sigmoid(o2).cpu().numpy())
                fold_p3.extend(torch.sigmoid(o3).cpu().numpy())
        
        final_probs_t1 += np.array(fold_p1)
        final_probs_t2 += np.array(fold_p2)
        final_probs_t3 += np.array(fold_p3)
        
        del model, optimizer
        torch.cuda.empty_cache()
        gc.collect()

    # --- CALCULATE OPTIMAL THRESHOLDS ---
    print("\nCalculating Optimal Thresholds (OOF)...")
    
    # Flatten validation lists
    oof_t1_flat = np.concatenate(oof_targets_t1)
    oof_p1_flat = np.concatenate(oof_preds_t1)
    
    oof_t2_flat = np.concatenate(oof_targets_t2)
    oof_p2_flat = np.concatenate(oof_preds_t2)
    
    oof_t3_flat = np.concatenate(oof_targets_t3)
    oof_p3_flat = np.concatenate(oof_preds_t3)

    # Find best threshold for T1
    thresh_t1, f1_t1 = get_best_threshold(oof_t1_flat, oof_p1_flat)
    print(f"Best Threshold T1: {thresh_t1:.2f} (Val F1: {f1_t1:.4f})")
    
    # Find best threshold for T2 (per label)
    thresh_t2 = []
    for i in range(len(Config.LABELS_T2)):
        th, score = get_best_threshold(oof_t2_flat[:, i], oof_p2_flat[:, i])
        thresh_t2.append(th)
        print(f"  T2-{Config.LABELS_T2[i]}: {th:.2f} (F1: {score:.4f})")
        
    # Find best threshold for T3 (per label)
    thresh_t3 = []
    for i in range(len(Config.LABELS_T3)):
        th, score = get_best_threshold(oof_t3_flat[:, i], oof_p3_flat[:, i])
        thresh_t3.append(th)
        print(f"  T3-{Config.LABELS_T3[i]}: {th:.2f} (F1: {score:.4f})")

    # --- APPLY THRESHOLDS TO TEST SUBMISSION ---
    print(f"\nGenerating Submission for {lang}...")
    
    avg_p1 = final_probs_t1 / n_folds
    avg_p2 = final_probs_t2 / n_folds
    avg_p3 = final_probs_t3 / n_folds
    
    # Apply dynamic thresholds
    pred_t1 = (avg_p1 >= thresh_t1).astype(int)
    
    pred_t2 = np.zeros_like(avg_p2, dtype=int)
    for i in range(len(Config.LABELS_T2)):
        pred_t2[:, i] = (avg_p2[:, i] >= thresh_t2[i]).astype(int)
        
    pred_t3 = np.zeros_like(avg_p3, dtype=int)
    for i in range(len(Config.LABELS_T3)):
        pred_t3[:, i] = (avg_p3[:, i] >= thresh_t3[i]).astype(int)
    
    # Post-Processing Consistency Rule
    for i in range(len(pred_t1)):
        if pred_t1[i][0] == 0:
            pred_t2[i, :] = 0
            pred_t3[i, :] = 0

    # Save
    ids_all = df_dev['id'].values
    
    # Save T1
    df_t1 = pd.DataFrame(pred_t1, columns=Config.LABELS_T1)
    df_t1.insert(0, 'id', ids_all)
    df_t1.to_csv(f'/kaggle/working/pred1_{lang}.csv', index=False)
    
    # Save T2
    df_t2 = pd.DataFrame(pred_t2, columns=Config.LABELS_T2)
    df_t2.insert(0, 'id', ids_all)
    df_t2.to_csv(f'/kaggle/working/pred2_{lang}.csv', index=False)
    
    # Save T3
    df_t3 = pd.DataFrame(pred_t3, columns=Config.LABELS_T3)
    df_t3.insert(0, 'id', ids_all)
    df_t3.to_csv(f'/kaggle/working/pred3_{lang}.csv', index=False)
    
    print(f"Files saved for {lang} using Optimized Thresholds and Focal Loss!")

In [30]:
run_kfold_pipeline(lang='eng', model_name='microsoft/deberta-v3-base', n_folds=3)


START IMPROVED PIPELINE FOR: ENG
MODEL: microsoft/deberta-v3-base


/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(



>>> Fold 1/3
   Epoch 1 Loss: 0.3168
   Epoch 2 Loss: 0.2339
   Epoch 3 Loss: 0.1728
   Epoch 4 Loss: 0.1351

>>> Fold 2/3
   Epoch 1 Loss: 0.3296
   Epoch 2 Loss: 0.2325
   Epoch 3 Loss: 0.1698
   Epoch 4 Loss: 0.1353

>>> Fold 3/3
   Epoch 1 Loss: 0.3215
   Epoch 2 Loss: 0.2313
   Epoch 3 Loss: 0.1690
   Epoch 4 Loss: 0.1453

Calculating Optimal Thresholds (OOF)...
Best Threshold T1: 0.48 (Val F1: 0.7491)
  T2-political: 0.39 (F1: 0.7419)
  T2-racial/ethnic: 0.38 (F1: 0.3901)
  T2-religious: 0.32 (F1: 0.2571)
  T2-gender/sexual: 0.31 (F1: 0.1299)
  T2-other: 0.33 (F1: 0.1714)
  T3-stereotype: 0.39 (F1: 0.4706)
  T3-vilification: 0.42 (F1: 0.6532)
  T3-dehumanization: 0.44 (F1: 0.4144)
  T3-extreme_language: 0.42 (F1: 0.5975)
  T3-lack_of_empathy: 0.36 (F1: 0.3444)
  T3-invalidation: 0.37 (F1: 0.4678)

Generating Submission for eng...
Files saved for eng using Optimized Thresholds and Focal Loss!


In [31]:
run_kfold_pipeline(lang='swa', model_name='Davlan/afro-xlmr-large', n_folds=5)


START IMPROVED PIPELINE FOR: SWA
MODEL: Davlan/afro-xlmr-large


tokenizer_config.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

   Epoch 1 Loss: 0.3739
   Epoch 2 Loss: 0.3752
   Epoch 3 Loss: 0.4111
   Epoch 4 Loss: 0.4062

>>> Fold 4/5


Some weights of XLMRobertaModel were not initialized from the model checkpoint at Davlan/afro-xlmr-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Epoch 1 Loss: 0.4182
   Epoch 2 Loss: 0.4072
   Epoch 3 Loss: 0.4051
   Epoch 4 Loss: 0.4011

>>> Fold 5/5


Some weights of XLMRobertaModel were not initialized from the model checkpoint at Davlan/afro-xlmr-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Epoch 1 Loss: 0.4134
   Epoch 2 Loss: 0.4054
   Epoch 3 Loss: 0.4016
   Epoch 4 Loss: 0.4010

Calculating Optimal Thresholds (OOF)...
Best Threshold T1: 0.45 (Val F1: 0.7112)
  T2-political: 0.30 (F1: 0.2879)
  T2-racial/ethnic: 0.42 (F1: 0.5891)
  T2-religious: 0.33 (F1: 0.4233)
  T2-gender/sexual: 0.28 (F1: 0.1431)
  T2-other: 0.34 (F1: 0.1799)
  T3-stereotype: 0.43 (F1: 0.6172)
  T3-vilification: 0.45 (F1: 0.6248)
  T3-dehumanization: 0.33 (F1: 0.2510)
  T3-extreme_language: 0.38 (F1: 0.4206)
  T3-lack_of_empathy: 0.41 (F1: 0.5008)
  T3-invalidation: 0.38 (F1: 0.4140)

Generating Submission for swa...
Files saved for swa using Optimized Thresholds and Focal Loss!
